In [ ]:
# --- Dependencies ---
# pip install torch torchvision torchaudio pandas numpy matplotlib scikit-learn tqdm pillow opencv-python

In [13]:
# --- Setup & Imports ---
import os, json, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from collections import Counter, defaultdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image, UnidentifiedImageError 

# Data Cleaning

In [10]:
# --- Paths ---
# Point to the folder that contains your images and the _annotations.coco.json
INPUT_JSON = Path("../data/labeled_images/_annotations.coco.json")
OUTPUT_JSON  = INPUT_JSON.with_name(INPUT_JSON.stem + "_cleaned.json")

def load(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def ensure_cat_name(coco, name, supercat="person-state"):
    """Ensure a category by name exists (lowercase match); return its id."""
    for c in coco.get("categories", []):
        if c["name"].lower() == name.lower():
            # also normalize the canonical name to lowercase for 'alert'
            if name == "alert":
                c["name"] = "alert"
            return c["id"]
    new_id = (max([c["id"] for c in coco.get("categories", [])] + [0]) + 1)
    coco.setdefault("categories", []).append({"id": new_id, "name": name, "supercategory": supercat})
    return new_id

def clean_annotations():
    coco = load(INPUT_JSON)
    coco.setdefault("images", [])
    coco.setdefault("annotations", [])
    coco.setdefault("categories", [])

    # Identify face category ids (case-insensitive)
    face_ids = {c["id"] for c in coco["categories"] if c.get("name","").lower() == "face"}

    # Ensure alert/drowsy categories exist
    alert_id  = ensure_cat_name(coco, "alert")
    drowsy_id = ensure_cat_name(coco, "drowsy")

    # Build image_id -> primary label; also track images to drop
    primary = {}
    drop_image_ids = set()

    for img in coco["images"]:
        tags = [t.lower() for t in img.get("extra", {}).get("user_tags", []) if isinstance(t, str)]
        tagset = set(tags)

        # If both, keep only 'alert'
        if "alert" in tagset and "drowsy" in tagset:
            tagset = {"alert"}

        # Drop images with only 'uncertain'
        if tagset == {"uncertain"}:
            drop_image_ids.add(img["id"])
            continue

        # Determine primary label, and write back normalized tags
        if "alert" in tagset:
            primary_label = "alert"
            img.setdefault("extra", {}).setdefault("user_tags", [])
            img["extra"]["user_tags"] = ["alert"]
        elif "drowsy" in tagset:
            primary_label = "drowsy"
            img.setdefault("extra", {}).setdefault("user_tags", [])
            img["extra"]["user_tags"] = ["drowsy"]
        else:
            primary_label = None
            # keep any non-uncertain leftovers (optional)
            img.setdefault("extra", {}).setdefault("user_tags", [])
            img["extra"]["user_tags"] = sorted(t for t in tagset if t != "uncertain")

        primary[img["id"]] = primary_label

    # Remove dropped images and their annotations
    if drop_image_ids:
        coco["images"] = [im for im in coco["images"] if im["id"] not in drop_image_ids]
        coco["annotations"] = [a for a in coco["annotations"] if a["image_id"] not in drop_image_ids]

    # Now, for each FACE annotation, switch ONLY the category_id (keep bbox etc identical)
    changed = 0
    for ann in coco["annotations"]:
        if ann.get("category_id") in face_ids:
            label = primary.get(ann["image_id"])
            if label == "alert":
                ann["category_id"] = alert_id
                changed += 1
            elif label == "drowsy":
                ann["category_id"] = drowsy_id
                changed += 1
            # If neither tag present, leave as 'face' (no other field changes)

    # Save
    src = Path(INPUT_JSON)
    out = Path(OUTPUT_JSON) if OUTPUT_JSON else src.with_name(src.stem + "_face_to_state.json")
    save(coco, out)

    print(f"Converted {changed} face annotations to 'alert'/'drowsy' while keeping bboxes identical.")
    print(f"Dropped {len(drop_image_ids)} images (only 'uncertain').")
    print(f"Output: {out}")

clean_annotations()

Converted 3159 face annotations to 'alert'/'drowsy' while keeping bboxes identical.
Dropped 0 images (only 'uncertain').
Output: ..\data\labeled_images\_annotations.coco_cleaned.json


In [11]:
# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

# Choose GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# ResNet-18 Classifier Model

In [ ]:
# ----------------------------
# Config (tweak as needed)
# ----------------------------
CFG = dict(
    # data
    context_scale=1.20,        # expand bbox by +20% around center to keep some context
    min_crop_size=8,           # safety clamp for tiny boxes
    img_size=224,              # model input size
    test_size=0.2,             # train/val split
    random_state=42,

    # optimization
    epochs=20,                 # total epochs (includes warmup)
    batch_size=32,
    num_workers=min(8, os.cpu_count() or 2),
    pin_memory=torch.cuda.is_available(),

    # fine-tuning schedule
    warmup_epochs=3,           # epochs training head-only before unfreezing layer4
    unfreeze_layer3_at=None,   # set to an int epoch to unfreeze layer3 too (e.g., 8); None = keep frozen

    # discriminative LRs (param groups)
    max_lr_head=1e-3,          # OneCycle max LR for head (higher)
    max_lr_layer4=3e-4,        # OneCycle max LR for layer4 (lower than head)
    max_lr_backbone=1e-4,      # OneCycle max LR for the rest backbone (lowest)

    weight_decay=1e-4,         # L2 regularization

    # regularization
    label_smoothing=0.05,      # CrossEntropy with label smoothing
    use_random_erasing=True,   # adds RandomErasing after ToTensor
    use_weighted_sampler=True, # handle class imbalance with WeightedRandomSampler

    # early stopping/checkpointing
    early_stop_patience=5,     # stop if val loss not improving
    artifacts_dir="artifacts"  # where to save weights/plots
)

# ----------------------------
# Small JSON helpers
# ----------------------------
def read_json(path):
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# ----------------------------
# Load cleaned COCO and index images
# ----------------------------
CLEAN_JSON = OUTPUT_JSON
coco = read_json(CLEAN_JSON)
data_root = INPUT_JSON.parent  # folder that contains the images referenced by file_name

def build_img_index(coco, data_root: Path):
    """
    image_id -> (abs_path, width, height)
    Warn if any images referenced by COCO don't exist on disk.
    """
    idx = {}
    missing = 0
    for img in coco.get("images", []):
        rel = Path(img.get("file_name", ""))
        abs_path = (data_root / rel).resolve()
        if abs_path.exists():
            idx[img["id"]] = (str(abs_path), img.get("width"), img.get("height"))
        else:
            missing += 1
    if missing:
        print(f"[WARN] Missing image files referenced by COCO: {missing}")
    return idx

img_index = build_img_index(coco, data_root)

# ----------------------------
# Build per-annotation samples for (7=alert, 8=drowsy)
# ----------------------------
name_to_id = {c["name"].lower(): c["id"] for c in coco.get("categories", [])}
ALERT_ID  = name_to_id.get("alert", 7)
DROWSY_ID = name_to_id.get("drowsy", 8)

samples = []  # each: (img_path, (x,y,w,h), y_int) where y_int: 0=drowsy, 1=alert
skipped = 0
for ann in coco.get("annotations", []):
    cid = ann.get("category_id")
    if cid not in (ALERT_ID, DROWSY_ID):
        continue
    img_id = ann.get("image_id")
    if img_id not in img_index:
        skipped += 1
        continue
    img_path, W, H = img_index[img_id]
    bbox = ann.get("bbox")
    if not bbox or len(bbox) != 4:
        skipped += 1
        continue
    y = 1 if cid == ALERT_ID else 0
    samples.append((img_path, tuple(bbox), y))

print(f"Total crops: {len(samples)}  |  skipped: {skipped}")
if not samples:
    raise RuntimeError("No crops found for category_id 7/8. Check your categories/annotations.")

# Label distribution (per-crop)
dist = Counter([s[2] for s in samples])
print("Per-crop label distribution (0=Drowsy, 1=Alert):", dict(dist))

# ----------------------------
# Dataset that crops on-the-fly
# ----------------------------
class CropDataset(Dataset):
    """
    On-the-fly crop from COCO bbox with optional context expansion.
    """
    def __init__(self, samples, transform=None, context_scale=1.20, min_size=8):
        self.samples = samples
        self.transform = transform
        self.context_scale = context_scale
        self.min_size = min_size

    def __len__(self): return len(self.samples)

    def _expand_and_clip(self, x, y, w, h, W, H):
        # Expand around center by context_scale (e.g., 1.2 adds 20% area)
        cx = x + w / 2.0
        cy = y + h / 2.0
        w2 = w * self.context_scale
        h2 = h * self.context_scale
        x1 = int(round(cx - w2 / 2.0))
        y1 = int(round(cy - h2 / 2.0))
        x2 = int(round(cx + w2 / 2.0))
        y2 = int(round(cy + h2 / 2.0))

        # Clip to image bounds
        x1 = max(0, min(x1, W - 1))
        y1 = max(0, min(y1, H - 1))
        x2 = max(1, min(x2, W))
        y2 = max(1, min(y2, H))

        # Enforce minimal crop size
        if x2 - x1 < self.min_size: x2 = min(W, x1 + self.min_size)
        if y2 - y1 < self.min_size: y2 = min(H, y1 + self.min_size)
        return x1, y1, x2, y2

    def __getitem__(self, idx):
        img_path, bbox, label = self.samples[idx]

        # Load image safely
        try:
            img = Image.open(img_path).convert("RGB")
        except (UnidentifiedImageError, OSError) as e:
            raise RuntimeError(f"Failed to open image: {img_path}") from e

        W, H = img.size
        x, y, w, h = bbox

        # Expand+clip the box (handles weird/floaty boxes robustly)
        x1, y1, x2, y2 = self._expand_and_clip(x, y, w, h, W, H)
        crop = img.crop((x1, y1, x2, y2))

        # Apply transforms
        if self.transform:
            crop = self.transform(crop)
        return crop, label

# ----------------------------
# Transforms (RandErasing optional)
# ----------------------------
def make_train_transform(img_size, use_random_erasing=True):
    t = [
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(7),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ]
    if use_random_erasing:
        # RandomErasing AFTER normalization is okay; torchvision applies on tensors
        t.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'))
    return transforms.Compose(t)

train_tfm = make_train_transform(CFG["img_size"], CFG["use_random_erasing"])
val_tfm = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ----------------------------
# Split (stratified per-crop)
# ----------------------------
all_idx = np.arange(len(samples))
y_all = [samples[i][2] for i in all_idx]
strat = y_all if len(set(y_all)) > 1 else None

tr_idx, va_idx = train_test_split(
    all_idx, test_size=CFG["test_size"], random_state=CFG["random_state"], stratify=strat
)

train_samples = [samples[i] for i in tr_idx]
val_samples   = [samples[i] for i in va_idx]
print(f"Train crops: {len(train_samples)} | Val crops: {len(val_samples)}")

# WeightedRandomSampler to mitigate class imbalance (optional)
if CFG["use_weighted_sampler"]:
    counts = Counter([s[2] for s in train_samples])
    class_weight = {c: 1.0 / counts[c] for c in counts}   # inverse frequency
    sample_weights = [class_weight[s[2]] for s in train_samples]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_samples), replacement=True)
    shuffle_flag = False
else:
    sampler = None
    shuffle_flag = True

# Datasets & loaders
train_ds = CropDataset(train_samples, transform=train_tfm,
                       context_scale=CFG["context_scale"], min_size=CFG["min_crop_size"])
val_ds   = CropDataset(val_samples, transform=val_tfm,
                       context_scale=CFG["context_scale"], min_size=CFG["min_crop_size"])

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=shuffle_flag, sampler=sampler,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    persistent_workers=(CFG["num_workers"] > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    persistent_workers=(CFG["num_workers"] > 0)
)

# ----------------------------
# Model + fine-tuning plan
# ----------------------------
def build_model():
    """
    ResNet-18 pretrained. Start with:
      • All backbone frozen
      • Replace FC with small head (256 → 2)
    We'll later unfreeze layer4 (and optionally layer3) with lower LR.
    """
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in m.parameters():
        p.requires_grad = False
    in_feats = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(in_feats, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(256, 2)        # 0=drowsy, 1=alert
    )
    return m

model = build_model().to(device)

# ----------------------------
# Loss: label smoothing (robustness to noisy labels)
# ----------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

# ----------------------------
# Param groups for discriminative LRs + OneCycleLR
#   • Group A: head (new FC)       → highest LR
#   • Group B: layer4 (deepest)    → medium LR (when unfrozen)
#   • Group C: rest of backbone    → lowest LR (when unfrozen)
# We'll start with only the HEAD trainable, then unfreeze progressively.
# ----------------------------
def param_groups(model, train_head=True, train_layer4=False, train_rest=False):
    groups = []
    # Head
    head_params = list(model.fc.parameters())
    for p in head_params: p.requires_grad = True
    groups.append({"params": head_params, "max_lr": CFG["max_lr_head"], "weight_decay": CFG["weight_decay"]})

    # Layer4
    if train_layer4:
        l4_params = list(model.layer4.parameters())
        for p in l4_params: p.requires_grad = True
        groups.append({"params": l4_params, "max_lr": CFG["max_lr_layer4"], "weight_decay": CFG["weight_decay"]})

    # Rest of backbone
    if train_rest:
        rest = []
        for blk in [model.layer3, model.layer2, model.layer1, model.conv1, model.bn1]:
            rest += list(blk.parameters())
        for p in rest: p.requires_grad = True
        groups.append({"params": rest, "max_lr": CFG["max_lr_backbone"], "weight_decay": CFG["weight_decay"]})

    return groups

# Initialize with HEAD only
optimizer = optim.AdamW(param_groups(model, train_head=True, train_layer4=False, train_rest=False), eps=1e-8)

# OneCycle needs total steps; we’ll recreate scheduler whenever we change param groups
def make_onecycle(optimizer, epochs, steps_per_epoch):
    # Collect per-group max_lrs from the param groups
    max_lrs = [g.get("max_lr", CFG["max_lr_backbone"]) for g in optimizer.param_groups]
    return optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lrs,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.15,            # warm fraction
        anneal_strategy='cos',     # cosine anneal
        div_factor=25.0,           # initial LR = max_lr/div_factor
        final_div_factor=1e4       # final LR = initial_lr/final_div_factor
    )

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ----------------------------
# Train / validate
# ----------------------------
def run_epoch(model, loader, optimizer, scaler, criterion, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="Train" if train else "Val", leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                logits = model(x)
                loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        total += y.size(0)
        correct += (pred == y).sum().item()

    return running_loss / total, correct / total

# ----------------------------
# Training loop with progressive unfreeze + early stopping + best checkpoint
# ----------------------------
best_val_loss = float("inf")
no_improve = 0

artifacts = Path(CFG["artifacts_dir"]); artifacts.mkdir(exist_ok=True)
best_path = artifacts / "best_drowsiness_resnet18_crops.pth"
last_path = artifacts / "last_drowsiness_resnet18_crops.pth"

# Create initial OneCycleLR for warm-up (head-only)
steps_per_epoch = max(1, math.ceil(len(train_loader.dataset) / CFG["batch_size"]))
scheduler = make_onecycle(optimizer, epochs=CFG["epochs"], steps_per_epoch=steps_per_epoch)

history = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}

for epoch in range(1, CFG["epochs"] + 1):
    # Unfreeze plan:
    #   • After warmup_epochs: unfreeze layer4 (deep features) with lower LR
    #   • Optionally later: unfreeze layer3 too
    if epoch == CFG["warmup_epochs"] + 1:
        # Rebuild optimizer with new param groups (head + layer4)
        optimizer = optim.AdamW(param_groups(model, train_head=True, train_layer4=True, train_rest=False), eps=1e-8)
        scheduler = make_onecycle(optimizer, epochs=CFG["epochs"] - (epoch-1), steps_per_epoch=steps_per_epoch)
        print(f"[Epoch {epoch}] Unfroze layer4 with discriminative LR.")

    if CFG["unfreeze_layer3_at"] and epoch == CFG["unfreeze_layer3_at"]:
        # Rebuild optimizer with (head + layer4 + rest part: layer3 added here)
        optimizer = optim.AdamW(param_groups(model, train_head=True, train_layer4=True, train_rest=True), eps=1e-8)
        scheduler = make_onecycle(optimizer, epochs=CFG["epochs"] - (epoch-1), steps_per_epoch=steps_per_epoch)
        print(f"[Epoch {epoch}] Unfroze layer3 (and the rest block group) with lower LR.")

    # Train
    tl, ta = run_epoch(model, train_loader, optimizer, scaler, criterion, device, train=True)
    scheduler.step()

    # Validate
    vl, va = run_epoch(model, val_loader, optimizer, scaler, criterion, device, train=False)

    # Log
    history["tr_loss"].append(tl); history["va_loss"].append(vl)
    history["tr_acc"].append(ta);  history["va_acc"].append(va)

    print(f"Epoch {epoch:02d}/{CFG['epochs']} | "
          f"Train Loss {tl:.4f} Acc {ta:.4f} | "
          f"Val Loss {vl:.4f} Acc {va:.4f}")

    # Early stopping on val loss + best checkpointing
    if vl < best_val_loss - 1e-6:
        best_val_loss = vl
        no_improve = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ↳ New best val loss. Saved: {best_path}")
    else:
        no_improve += 1
        if no_improve >= CFG["early_stop_patience"]:
            print(f"  ↳ Early stopping (no improvement for {CFG['early_stop_patience']} epochs).")
            break

# Save last model too (useful if early stop triggers)
torch.save(model.state_dict(), last_path)
print(f"Saved last model: {last_path}")

# Save label map for inference
write_json({"0": "Drowsy", "1": "Alert"}, artifacts / "class_map.json")

# ----------------------------
# Plot curves
# ----------------------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history["tr_loss"], label="Train Loss")
plt.plot(history["va_loss"], label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history["tr_acc"], label="Train Acc")
plt.plot(history["va_acc"], label="Val Acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy"); plt.legend()

plt.tight_layout()
plt.show()

print(f"Best weights → {best_path}")
print(f"Class map   → {artifacts / 'class_map.json'}")

C:\Users\Sebert\AppData\Local\Temp\ipykernel_5904\3811067723.py:317: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Total crops: 3159  |  skipped: 0
Per-crop label distribution (0=Drowsy, 1=Alert): {0: 1560, 1: 1599}
Train crops: 2527 | Val crops: 632


Train:   0%|          | 0/79 [00:00<?, ?it/s]